# chesszero — étape 4b : essai de débit puis pré-entraînement sur Kaggle

**Réglages à faire AVANT de lancer** (panneau de droite, *Session options*) :

| Réglage | Valeur |
|---|---|
| Accelerator | **GPU T4 x2** (le code n'en utilise qu'un ; le T4 a des tensor cores, donc la précision mixte fp16 y est rapide). À défaut **GPU P100**. |
| Internet | **On** (pour `git clone` et `pip install`). Demande un compte vérifié par téléphone. |
| Persistence | *No persistence* (les checkpoints passent par la sortie du notebook, voir plus bas) |
| Input | **Add Input** → *Datasets* → ton Dataset `lichess-24m` (le `.zip` est déjà décompressé par Kaggle) |

**Contenu** : 1) clone + installation sans toucher à torch ; 2) vérification de l'environnement ;
3) localisation du Dataset ; 4) reprise d'un `last.pt` d'une session précédente ; 5) essai de débit de 10 min
avec les masques, 6) essai de 4 min sans masque, comparaison ; 7) entraînement complet (désactivé par défaut).

**Reprise entre deux sessions** : `/kaggle/working` est effacé à la fin de chaque session. Ce qui survit, c'est
la **sortie d'une version sauvegardée** (*Save Version → Save & Run All (Commit)*). Pour reprendre :
ouvre le notebook, **Add Input → Notebooks (ou « Your Work ») → ce même notebook** ; sa dernière sortie apparaît
sous `/kaggle/input/...`, et la cellule 4 y retrouve `kaggle_sup/last.pt`, `best.pt` et `metrics.csv`
et les copie dans `/kaggle/working/kaggle_sup`, d'où l'entraînement repart au bon pas (reprise identique à un run
sans coupure). Détail dans `docs/journal.md`.

In [ ]:
# 0. Réglages et utilitaires -------------------------------------------------------------------
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Enz091/Project_chess.git"
CONFIG = "kaggle"                      # configs/kaggle.toml (10 blocs x 128)
INPUT_ROOT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
REPO_DIR = WORKING / "Project_chess"
OUT = WORKING / "kaggle_sup"           # NE PAS renommer : la cellule de reprise cherche « kaggle_sup/last.pt »

RUN_BENCH = True                       # essai de débit (10 min avec masques)
BENCH_MINUTES = 10
NOMASK_MINUTES = 4                     # essai sans masque pour comparer (0 = sauter)
RUN_FULL = False                       # True pour lancer le vrai entraînement (et RUN_BENCH = False)
FULL_MINUTES = 660                     # 11 h : s'arrête proprement avant la limite de 12 h de Kaggle
NO_MASK_LOSS = False                   # True si l'essai montre que le GPU attend les masques
ALLOW_CPU = False                      # True seulement pour déboguer sans GPU
MANAGE_REPO = True                     # False : ne pas cloner ni installer (REPO_DIR déjà prêt)


def run(cmd, cwd=None):
    # Lance une commande et affiche sa sortie au fil de l'eau ; lève une erreur si elle échoue.
    proc = subprocess.Popen([str(c) for c in cmd], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"échec ({proc.returncode}) : {' '.join(str(c) for c in cmd)}")

In [ ]:
# 1. Dépôt et installation (sans réinstaller torch) -----------------------------------------------
import importlib.metadata as md

torch_before = md.version("torch")
if MANAGE_REPO:
    if (REPO_DIR / ".git").exists():
        run(["git", "pull", "--ff-only"], cwd=REPO_DIR)
    else:
        run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
    # Pas de torch dans les dépendances du projet : pip garde le torch CUDA fourni par Kaggle.
    run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR])
run(["git", "log", "-1", "--format=commit %h  %s"], cwd=REPO_DIR)
torch_after = md.version("torch")
print(f"torch : {torch_before} -> {torch_after}")
assert torch_before == torch_after, "pip a modifié torch : à ne pas faire sur Kaggle"

In [ ]:
# 2. Vérification de l'environnement ----------------------------------------------------------------
import platform

import torch

assert sys.version_info >= (3, 11), f"Python >= 3.11 requis (tomllib), trouvé {platform.python_version()}"
print("Python           :", platform.python_version(), "|", platform.platform())
print("torch            :", torch.__version__, "| CUDA du build :", torch.version.cuda)
cores = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
print("Cœurs CPU        :", cores)
WORKERS = max(1, min(3, cores - 1))   # 1 cœur pour le processus d'entraînement, le reste pour les masques
print("Workers de masque:", WORKERS)
if Path("/proc/meminfo").exists():
    mem = {l.split(":")[0]: int(l.split()[1]) for l in Path("/proc/meminfo").read_text().splitlines()[:3]}
    print(f"RAM              : {mem['MemTotal'] / 1e6:.0f} Go (libre : {mem['MemAvailable'] / 1e6:.0f} Go)")
print(f"Disque /working  : {shutil.disk_usage(WORKING).free / 1e9:.0f} Go libres")

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU              : {p.name} | {p.total_memory / 1e9:.0f} Go | capacité {p.major}.{p.minor} | {torch.cuda.device_count()} visible(s)")
    # Précision mixte : une convolution sous autocast doit sortir en float16
    conv = torch.nn.Conv2d(128, 128, 3, padding=1).cuda()
    with torch.autocast("cuda", dtype=torch.float16):
        y = conv(torch.randn(64, 128, 8, 8, device="cuda"))
    print("Précision mixte  :", y.dtype, "(attendu : torch.float16)")
    assert y.dtype == torch.float16
    if shutil.which("nvidia-smi"):
        run(["nvidia-smi", "--query-gpu=name,driver_version,utilization.gpu,memory.used,memory.total", "--format=csv"])
else:
    msg = "Aucun GPU : activer l'accélérateur (Session options -> Accelerator -> GPU T4 x2)."
    if not ALLOW_CPU:
        raise RuntimeError(msg)
    print("ATTENTION :", msg)

In [ ]:
# 3. Dataset ------------------------------------------------------------------------------------
import json

candidates = sorted(INPUT_ROOT.glob("**/train_00000.npz"))
if not candidates:
    raise FileNotFoundError(f"Aucun shard train_00000.npz sous {INPUT_ROOT} : ajouter le Dataset (Add Input).")
DATA = candidates[0].parent
if len(candidates) > 1:
    print("Plusieurs datasets trouvés, le premier est utilisé :", [str(c.parent) for c in candidates])
manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))
print("Dataset :", DATA)
print(f"Positions : train {manifest['positions']['train']:,} | val {manifest['positions']['val']:,} | "
      f"{len(list(DATA.glob('train_*.npz')))} shards d'entraînement")

In [ ]:
# 4. Reprise depuis la sortie d'une session précédente --------------------------------------------------
def step_of(path):
    return torch.load(path, map_location="cpu", weights_only=False)["step"]

previous = sorted(INPUT_ROOT.glob("**/kaggle_sup/last.pt"))
if (OUT / "last.pt").exists():
    print("last.pt déjà présent dans", OUT, "-> reprise au pas", step_of(OUT / "last.pt"))
elif previous:
    best = max(previous, key=step_of)
    OUT.mkdir(parents=True, exist_ok=True)
    for name in ("last.pt", "best.pt", "metrics.csv"):
        if (best.parent / name).exists():
            shutil.copy2(best.parent / name, OUT / name)
    print(f"Checkpoint restauré depuis {best.parent} (pas {step_of(OUT / 'last.pt')})")
else:
    print("Aucun checkpoint précédent : l'entraînement partira de zéro.")

In [ ]:
# 5. Essai de débit : config kaggle avec masques, 10 minutes -----------------------------------------------
BENCH, BENCH_NOMASK = WORKING / "bench", WORKING / "bench_nomask"
TRAIN = REPO_DIR / "scripts" / "train.py"
SUMMARY = REPO_DIR / "scripts" / "summarize_run.py"

if RUN_BENCH:
    shutil.rmtree(BENCH, ignore_errors=True)   # toujours repartir de zéro : on mesure, on n'entraîne pas pour de bon
    run([sys.executable, TRAIN, "--config", CONFIG, "--data", DATA, "--out", BENCH,
         "--workers", WORKERS, "--max-minutes", BENCH_MINUTES])
    print("\n================ RÉSUMÉ (avec masques) ================")
    run([sys.executable, SUMMARY, BENCH])

In [ ]:
# 6. Même essai SANS masque (perte non masquée) pour chiffrer le gain ---------------------------------------------
if RUN_BENCH and NOMASK_MINUTES:
    shutil.rmtree(BENCH_NOMASK, ignore_errors=True)
    run([sys.executable, TRAIN, "--config", CONFIG, "--data", DATA, "--out", BENCH_NOMASK,
         "--workers", WORKERS, "--max-minutes", NOMASK_MINUTES, "--no-mask-loss"])
    print("\n================ RÉSUMÉ (sans masque) ================")
    run([sys.executable, SUMMARY, BENCH_NOMASK])

    import statistics
    from chesszero.train import read_metrics

    def throughput(d):
        rows = [r for r in read_metrics(d / "metrics.csv") if r["split"] == "train"]
        rows = rows[2:] or rows
        return statistics.fmean(r["pos_per_s"] for r in rows)

    with_mask, without = throughput(BENCH), throughput(BENCH_NOMASK)
    gain = without / with_mask - 1
    print(f"\nDébit avec masques : {with_mask:,.0f} pos/s | sans masque : {without:,.0f} pos/s | gain : {gain:+.0%}")
    print("-> activer la perte sans masque (NO_MASK_LOSS = True)" if gain > 0.10 else
          "-> gain faible : garder les masques (le GPU n'attend pas les masques)")

# On ne garde pas les checkpoints d'essai (ils ne doivent surtout pas être pris pour une reprise) :
for d in (BENCH, BENCH_NOMASK):
    for f in d.glob("*.pt"):
        f.unlink()

In [ ]:
# 7. Entraînement complet (RUN_FULL = True) -------------------------------------------------------------------
# Reprend tout seul depuis OUT/last.pt s'il existe. S'arrête proprement après FULL_MINUTES minutes
# d'entraînement ; relancer dans une nouvelle session (cellule 4) pour continuer.
if RUN_FULL:
    cmd = [sys.executable, TRAIN, "--config", CONFIG, "--data", DATA, "--out", OUT,
           "--workers", WORKERS, "--max-minutes", FULL_MINUTES]
    if NO_MASK_LOSS:
        cmd.append("--no-mask-loss")
    run(cmd)
    print("\n================ RÉSUMÉ DU DÉBIT ================")
    run([sys.executable, SUMMARY, OUT, "--skip", 5])
else:
    print("RUN_FULL = False : entraînement complet non lancé.")